# Simulazione di ascesa e guida di un lanciatore multistadio — Demo Colab

Questo notebook esegue l'intera pipeline del progetto
[launch-vehicle-ascent-guidance](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance)
direttamente nel browser, senza alcun setup locale: clona il repository,
installa le dipendenze, esegue la simulazione (gravity turn + guida a
tangente lineare, caso di validazione Falcon 9 verso un'orbita LEO a
200&nbsp;km) e mostra grafici e animazioni direttamente qui sotto.

**Nessun calcolo nuovo**: questo notebook richiama solo le funzioni già
presenti in `lanciatore/`, esattamente come documentato in
[README.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/README.md)
e [VALIDATION.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/VALIDATION.md).
Il progetto simula un'unica configurazione (lanciatore classe Falcon 9,
orbita target LEO 200&nbsp;km) — non c'è nulla da scegliere: basta eseguire
tutte le celle in ordine (**Runtime → Esegui tutto**).

Per il dettaglio dei limiti del modello e delle approssimazioni
dichiarate, vedi [VALIDATION.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/VALIDATION.md);
per il log completo di sviluppo, [STATUS.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/STATUS.md).

## 1. Setup: clona il repository e installa le dipendenze

In [ ]:
import os

REPO_URL = "https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance.git"
REPO_DIR = "launch-vehicle-ascent-guidance"

# Cella pensata per essere ri-eseguibile senza errori (es. con "Esegui
# tutto" ripetuto sullo stesso runtime, senza restart): clona solo se la
# cartella non esiste ancora, e cambia directory solo se non ci si e'
# gia' dentro.
if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}

print("Directory di lavoro:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from IPython.display import Image, display

from lanciatore import validazione as val
from lanciatore import visualizzazione as viz

## 2. Esecuzione della pipeline di validazione

`esegui_validazione()` è la funzione che orchestra l'intera simulazione
(gravity turn dello Stadio 1, staging, guida a tangente lineare dello
Stadio 2, calcolo del delta-v ideale e scomposizione delle perdite) con
i parametri di default del progetto — nessun argomento custom, nessun
ricalcolo qui: gli stessi numeri già documentati in STATUS.md/VALIDATION.md.

In [ ]:
risultato = val.esegui_validazione()

print(viz.riepilogo_testuale(risultato))

## 3. Verifica di stabilità orbitale dello stato finale

Il check di delta-v sopra verifica la *capacità ideale* del veicolo, non
se la traiettoria integrata raggiunga davvero un'orbita stabile. Questo
progetto esclude per scope la rotazione terrestre (dichiarato fin
dall'inizio in CLAUDE.md): senza quel bonus di velocità, il perigeo
risulta sotto la superficie terrestre — un limite noto e già quantificato
(vedi VALIDATION.md), non un errore nascosto. Con il bonus di rotazione
(già quantificato, `≈409 m/s` per un sito equatoriale/subtropicale),
l'orbita diventa stabile.

In [ ]:
assemblaggio = risultato["assemblaggio"]
orbita_senza = assemblaggio["orbita_senza_bonus"]
orbita_con = assemblaggio["orbita_con_bonus"]

print("Orbita SENZA bonus di rotazione terrestre (caso rigoroso, scope dichiarato in CLAUDE.md):")
print(
    f"  perigeo = {orbita_senza['perigeo'] / 1000:.2f} km, "
    f"apogeo = {orbita_senza['apogeo'] / 1000:.2f} km, "
    f"orbita stabile: {orbita_senza['perigeo_valido']}"
)
print()
print(
    f"Orbita CON bonus di rotazione terrestre "
    f"({val.BONUS_ROTAZIONE_TERRESTRE:.2f} m/s, gia' quantificato in CLAUDE.md/VALIDATION.md):"
)
print(
    f"  perigeo = {orbita_con['perigeo'] / 1000:.2f} km, "
    f"apogeo = {orbita_con['apogeo'] / 1000:.2f} km, "
    f"orbita stabile: {orbita_con['perigeo_valido']}"
)

## 4. Grafici e animazioni

Generati con le stesse funzioni di `lanciatore/visualizzazione.py` usate
in locale (`python -m lanciatore.visualizzazione`), qui mostrate
direttamente nel notebook invece che solo salvate su disco.

In [ ]:
serie = viz.estrai_serie_temporali(assemblaggio)
serie_interpolata = viz.estrai_serie_temporali_interpolata(assemblaggio)

percorso_grafico_tempo = "output/quota_velocita_tempo.png"
percorso_grafico_traiettoria = "output/traiettoria.png"
percorso_animazione = "output/traiettoria.gif"
percorso_animazione_avanzata = "output/traiettoria_avanzata.gif"

viz.grafico_quota_velocita_tempo(serie, percorso_grafico_tempo)
viz.grafico_traiettoria(serie, percorso_grafico_traiettoria)
viz.anima_traiettoria(serie, percorso_animazione)
viz.anima_traiettoria_avanzata(serie_interpolata, percorso_animazione_avanzata)

print("File generati:")
for percorso in (
    percorso_grafico_tempo,
    percorso_grafico_traiettoria,
    percorso_animazione,
    percorso_animazione_avanzata,
):
    print(" ", percorso)

### Quota, velocità e massa nel tempo

In [ ]:
display(Image(filename=percorso_grafico_tempo))

### Traiettoria nel piano (x, h)

In [ ]:
display(Image(filename=percorso_grafico_traiettoria))

### Animazione semplice

In [ ]:
display(Image(filename=percorso_animazione))

### Animazione avanzata (razzo orientato, scia, stage separation)

Il simbolo del razzo ruota seguendo l'angolo di volo reale, la scia
cambia colore esattamente all'istante di separazione degli stadi.
Nessuna fisica nuova: solo rendering di numeri già calcolati sopra
(vedi `lanciatore.visualizzazione.angolo_volo_gradi`).

In [ ]:
display(Image(filename=percorso_animazione_avanzata))

## Per approfondire

- [README.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/README.md) — panoramica del progetto
- [VALIDATION.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/VALIDATION.md) — limiti del modello, impatti misurati, confronto con progetti di riferimento
- [STATUS.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/STATUS.md) — log completo di sviluppo, ciclo per ciclo
- [CLAUDE.md](https://github.com/ValerioPeperoni/launch-vehicle-ascent-guidance/blob/master/CLAUDE.md) — scope del progetto e vincoli tecnici